# CBB Team Clustering Models

### Goal: Use K-means and K-nearest neighbors models to compare teams and identify team archetypes

All logic lives in the `cbb_similarity` package (`src/cbb_similarity/`). This notebook is for exploration and presentation. Install the package once with `pip install -e ".[dev]"` from the project root.

## Imports

In [1]:
from cbb_similarity.config import ARCHETYPE_SPECS, METRIC_GROUPS
from cbb_similarity.data import load_stat_labels, load_stats, resolve_season
from cbb_similarity.models import generate_comps_table
from cbb_similarity.pipeline import build_features, fit_archetypes
from cbb_similarity.visualization import plot_archetypes, plot_radar_comparison

## Load data and build features
Metrics are standardized within each season (model inputs) and percentile-ranked within each season (for display).

In [2]:
stats = load_stats()
stat_labels = load_stat_labels()
features = build_features(stats)
season = resolve_season(stats, "2025-26")

## KNN similarity: overall comps

In [3]:
generate_comps_table(features.scaled, features.ranks, "Michigan", season, METRIC_GROUPS["overall"])

,Team,similarity,hexColor1,hexColor2,stlPct,blkPct,pfPct,atr2FgaFreq,atr2FgPct,freqChncTr,...,pace,ftaRate,tovPct,fg3Pct,lane2FgPctAgst,lane2FgaFreqAgst,mid2FgPctAgst,mid2FgaFreqAgst,fga3RateAgst,fg3PctAgst
0,Michigan 2025-26,1.000000,#fcd414,#FAFAFA,0.123288,0.994521,0.063014,0.756164,0.928767,0.958904,...,0.830137,0.747945,0.465753,0.912329,0.030137,0.052055,0.079452,0.947945,0.758904,0.063014
1,Gonzaga 2021-22,0.835246,#babac5,#22305f,0.298883,0.932961,0.078212,0.938547,0.986034,0.997207,...,0.977654,0.569832,0.108939,0.907821,0.005587,0.715084,0.178771,0.874302,0.134078,0.106145
2,Duke 2024-25,0.769053,#1c4c9c,#204c9c,0.560440,0.733516,0.368132,0.668956,0.851648,0.508242,...,0.228022,0.502747,0.038462,0.969780,0.019231,0.112637,0.097527,0.989011,0.456044,0.134615
3,Kansas 2021-22,0.758132,#f09716,#2769b7,0.368715,0.770950,0.256983,0.632682,0.768156,0.754190,...,0.689944,0.709497,0.243017,0.854749,0.279330,0.462291,0.276536,0.912011,0.231844,0.067039
4,Arizona 2021-22,0.756722,#c8394f,#202f56,0.444134,0.930168,0.270950,0.994413,0.924581,0.935754,...,0.938547,0.846369,0.449721,0.729050,0.008380,0.446927,0.150838,0.918994,0.226257,0.388268
5,Duke 2025-26,0.736037,#1c4c9c,#204c9c,0.758904,0.676712,0.178082,0.852055,0.882192,0.383562,...,0.282192,0.764384,0.313699,0.608219,0.084932,0.035616,0.282192,0.746575,0.920548,0.093151
6,Arizona 2022-23,0.727130,#c8394f,#202f56,0.258953,0.396694,0.290634,0.764463,0.969697,0.928375,...,0.950413,0.853994,0.634986,0.939394,0.286501,0.033058,0.409091,0.988981,0.641873,0.247934
7,Michigan 2024-25,0.722463,#fcd414,#FAFAFA,0.085165,0.535714,0.442308,0.804945,0.619505,0.857143,...,0.837912,0.748626,0.906593,0.381868,0.184066,0.101648,0.247253,0.978022,0.532967,0.090659
8,Virginia 2025-26,0.699072,#dd572c,#24345a,0.454795,0.997260,0.597260,0.564384,0.473973,0.639726,...,0.191781,0.323288,0.284932,0.865753,0.016438,0.438356,0.523288,0.860274,0.339726,0.147945
9,Georgia St. 2020-21,0.680417,#254594,#c9d1e4,0.657061,0.893372,0.194524,0.403458,0.824207,0.922190,...,0.723343,0.440922,0.092219,0.775216,0.213256,0.194524,0.242075,0.491354,0.801153,0.553314


## Fit archetype models (shooting, playstyle, defense)

In [4]:
results = fit_archetypes(features)
results.assignments.head()

,teamId,competitionId,teamMarket,teamName,competitionName,hexColor1,hexColor2,shooting_cluster,shooting_archetype,shooting_x,shooting_y,playstyle_cluster,playstyle_archetype,playstyle_x,playstyle_y,defense_cluster,defense_archetype,defense_x,defense_y
0,103680,33533,Hartford,Hawks,2022-23 Men's Basketball,#000000,#000000,2,Elite Shooting,-0.124274,-1.572094,3,Slow and Physical,-1.130876,1.757470,0,Paint Containment Defense,-0.053133,0.354798
1,103454,33533,Chicago St.,Cougars,2022-23 Men's Basketball,#194624,#c0c0c0,2,Elite Shooting,1.092855,-2.882874,2,Inefficient Isolation,0.688458,-0.494667,2,Conservative Perimeter Defense,-0.438768,0.543383
2,103454,36046,Chicago St.,Cougars,2023-24 Men's Basketball,#194624,#c0c0c0,2,Elite Shooting,-1.037410,-2.302311,0,Controlled Ball Movement,1.445534,-0.713909,1,Turnover-Forcing Pressure,2.124678,-1.419620
3,104370,27693,UNC Asheville,Bulldogs,2020-21 Men's Basketball,#3d4b6a,#cbc8ca,2,Elite Shooting,1.628195,-2.639403,1,High-Tempo Aggressive,0.353967,0.279354,3,Disruptive Rim Protection,0.463291,-3.513269
4,103446,27693,Charleston So.,Buccaneers,2020-21 Men's Basketball,#24447c,#28447c,2,Elite Shooting,-0.419887,-0.210369,0,Controlled Ball Movement,0.874777,0.561067,2,Conservative Perimeter Defense,-0.038864,-0.692747


In [5]:
def show_archetypes(name):
    spec = ARCHETYPE_SPECS[name]
    for i, comp in enumerate(results.models[name].top_component_features(), start=1):
        print(f"PC{i}:", comp)
    plot_archetypes(
        results.assignments, x_col=f"{name}_x", y_col=f"{name}_y", color_col=f"{name}_archetype",
        x_title=spec.x_title, y_title=spec.y_title, title=spec.plot_title,
        hover_cols=["teamMarket", "competitionName"],
    ).show()
    return results.profiles[name].round(2)


def show_comps(team, name, n_teams=3):
    metrics = list(METRIC_GROUPS[name])
    comps = generate_comps_table(features.scaled, features.ranks, team, season, metrics)
    title = f"{comps['Team'].iloc[0]} {name.title()} Comparison"
    plot_radar_comparison(comps, metrics, stat_labels, n_teams=n_teams, title=title).show()
    return comps.head(n_teams)

## Shooting Style KMeans model

In [6]:
show_archetypes("shooting")

PC1: [('fga3Rate', 0.5589197201011152), ('mid2FgaFreq', -0.5088414827965503)]
PC2: [('paint2FgaFreq', 0.49259589460482384), ('atr2FgaFreq', -0.47942345102171313)]


,n_teams,atr2FgaFreq,atr2FgPct,paint2FgPct,paint2FgaFreq,mid2FgPct,mid2FgaFreq,fga3Rate,fg3Pct
shooting_archetype,,,,,,,,,
Elite Shooting,579,0.71,0.39,0.35,0.32,0.31,0.30,0.62,0.45
Midrange Heavy,486,0.29,0.62,0.71,0.84,0.49,0.56,0.40,0.52
Paint Dominant,538,0.51,0.33,0.34,0.53,0.51,0.79,0.25,0.33
Rim and Three,556,0.46,0.67,0.64,0.38,0.70,0.38,0.72,0.71


In [7]:
show_comps("Duke", "shooting")

,Team,similarity,hexColor1,hexColor2,atr2FgaFreq,atr2FgPct,paint2FgPct,paint2FgaFreq,mid2FgPct,mid2FgaFreq,fga3Rate,fg3Pct
0,Duke 2025-26,1.000000,#1c4c9c,#204c9c,0.852055,0.882192,0.864384,0.263014,0.852055,0.054795,0.734247,0.608219
1,Utah St. 2024-25,0.963706,#ada093,#162535,0.868132,0.755495,0.898352,0.351648,0.817308,0.056319,0.747253,0.782967
2,Southern Utah 2020-21,0.947246,#2d2229,#c92424,0.829971,0.850144,0.861671,0.256484,0.582133,0.100865,0.835735,0.561960


## Playstyle KMeans model

In [8]:
show_archetypes("playstyle")

PC1: [('ftaRate', 0.6798621214387163), ('pace', 0.46065303073203107)]
PC2: [('tovPct', 0.6045031534195086), ('orbPct', -0.5472775853063149)]


,n_teams,astPct,tovPct,pace,orbPct,ftaRate
playstyle_archetype,,,,,,
Controlled Ball Movement,535,0.52,0.75,0.70,0.58,0.77
High-Tempo Aggressive,547,0.25,0.35,0.60,0.37,0.47
Inefficient Isolation,565,0.53,0.40,0.34,0.78,0.49
Slow and Physical,512,0.72,0.52,0.37,0.25,0.28


In [9]:
show_comps("Duke", "playstyle")

,Team,similarity,hexColor1,hexColor2,astPct,tovPct,pace,orbPct,ftaRate
0,Duke 2025-26,1.000000,#1c4c9c,#204c9c,0.865753,0.313699,0.282192,0.983562,0.764384
1,Purdue 2022-23,0.972309,#20201f,#c8b487,0.966942,0.212121,0.060606,0.994490,0.900826
2,Southern California 2021-22,0.965127,#fac01c,#a41c33,0.659218,0.343575,0.370112,0.896648,0.586592


## Defense Style KMeans model

In [10]:
show_archetypes("defense")

PC1: [('stlPct', 0.5917523973630316), ('tovPctAgst', 0.5737424521325382)]
PC2: [('fga3RateAgst', 0.630343096279464), ('lane2FgaFreqAgst', -0.59802127739784)]


,n_teams,blkPct,stlPct,drbPct,pfPct,tovPctAgst,lane2FgPctAgst,lane2FgaFreqAgst,fga3RateAgst,fg3PctAgst
defense_archetype,,,,,,,,,,
Conservative Perimeter Defense,662,0.34,0.27,0.55,0.49,0.29,0.58,0.69,0.30,0.57
Disruptive Rim Protection,477,0.81,0.59,0.52,0.48,0.54,0.21,0.54,0.42,0.30
Paint Containment Defense,548,0.42,0.46,0.60,0.38,0.44,0.57,0.20,0.81,0.55
Turnover-Forcing Pressure,472,0.51,0.80,0.30,0.69,0.82,0.61,0.54,0.51,0.55


In [11]:
show_comps("Duke", "defense")

,Team,similarity,hexColor1,hexColor2,blkPct,stlPct,drbPct,pfPct,tovPctAgst,lane2FgPctAgst,lane2FgaFreqAgst,fga3RateAgst,fg3PctAgst
0,Duke 2025-26,1.000000,#1c4c9c,#204c9c,0.676712,0.758904,0.964384,0.178082,0.608219,0.084932,0.035616,0.920548,0.093151
1,Clemson 2025-26,0.903461,#f4541c,#f8541c,0.454795,0.484932,0.912329,0.397260,0.656164,0.246575,0.149315,0.830137,0.279452
2,Gonzaga 2025-26,0.900709,#babac5,#22305f,0.756164,0.895890,0.939726,0.219178,0.920548,0.106849,0.101370,0.794521,0.076712
